In [ ]:
#libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#loading the raw dataset
retail_df = pd.read_excel("/Users/manesh/Documents/Manesh /Masters/Portfolios/uk-retail-customer-analytics/Data/Online Retail.xlsx")

print("Raw Dataset Shape: ", retail_df.shape)

#Record the pre-cleaning baseline
initial_rows = len(retail_df)
initial_columns = retail_df.shape[1]
initial_missings = retail_df.isnull().sum().sum()
initial_duplicates = retail_df.duplicated().sum()

print("Initial rows:", initial_rows)
print("Initial columns:", initial_columns)
print("Initial missing values:", initial_missings)
print("Initial duplicates:", initial_duplicates)

#correct the datatypes 
retail_df["InvoiceDate"] = pd.to_datetime(
    retail_df["InvoiceDate"], 
    errors="coerce"
)
#CustomerID data type conversion is handling after removing the missing values 

#Handling Missing values 
retail_df["Description"] = (
    retail_df["Description"]
    .fillna("Unknown")
)

retail_df = retail_df.dropna(
    subset=["CustomerID"] 
)

#remove the confirmed exact duplicates 
duplicates_before = retail_df.duplicated().sum()

retail_df = retail_df.drop_duplicates(
    keep="first"
).reset_index(drop=True)

print("duplicates removed:", duplicates_before)

#standadize categorical/text data 
retail_df["Description"] = (
  retail_df["Description"]
  .astype(str)
  .str.strip()
 )

retail_df["Country"] = (
    retail_df["Country"]
    .astype(str)
    .str.strip()
)

#verify the categories 
print(retail_df["Country"].unique())

#Handle cancellations / returns
retail_df["TransactionStatus"] = np.where(
    retail_df["InvoiceNo"]
    .astype(str)
    .str.startswith("C"),
    "Cancelled",
    "Completed"
)

#Handling Customer ID values  ================

#  Handle Missing CustomerID
missing_customer_id_before = retail_df["CustomerID"].isna().sum()

print(
    "Missing CustomerID before cleaning:",
    missing_customer_id_before
)

# Remove records without CustomerID
retail_df = retail_df.dropna(
    subset=["CustomerID"]
).copy()

# Reset index
retail_df.reset_index(
    drop=True,
    inplace=True
)

#Validate the missing customer ID removal
print(
    "Missing CustomerID after cleaning:",
    retail_df["CustomerID"].isna().sum()
)

print(
    "Rows removed:",
    missing_customer_id_before
)

print(
    "Dataset shape after CustomerID cleaning:",
    retail_df.shape
)

#Coversion of CustomerID
retail_df["CustomerID"] = (
    retail_df["CustomerID"]
    .astype("Int64")
)

#Check the conversion 
print(retail_df["CustomerID"].dtype)


#Outlier analysis and treatment ===================
retail_df[["Quantity", "UnitPrice"]].describe()

# Investigate Extreme Quantity Values
quantity_extremes = retail_df[
    (retail_df["Quantity"] == retail_df["Quantity"].max()) |
    (retail_df["Quantity"] == retail_df["Quantity"].min())
]

display(quantity_extremes)

#largest possitive transaction 

display(
    retail_df.nlargest(
        20,
        "Quantity"
    )[
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "UnitPrice",
            "CustomerID",
            "InvoiceDate"
        ]
    ]
)

#Largest negative transactions 

display(
    retail_df.nsmallest(
        20,
        "Quantity"
    )[
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "UnitPrice",
            "CustomerID",
            "InvoiceDate"
        ]
    ]
)

# Investigate whether positive and negative extremes are connected
extreme_quantity = 80995

display(
    retail_df[
        retail_df["Quantity"].abs() == extreme_quantity
    ][
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "UnitPrice",
            "CustomerID",
            "InvoiceDate"
        ]
    ]
)

# Investigate UnitPrice = 0
zero_price = retail_df[
    retail_df["UnitPrice"] == 0 
]

print(
    "Zero Price Transactions: ",
    len(zero_price)
)

display(
    zero_price[
        [
             "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "UnitPrice",
            "CustomerID",
            "InvoiceDate"

        ]
    ].head(50)
)

display(
   zero_price["Description"] 
   .value_counts()
   .head(30)
)

#Investigate the maximum UnitPrice of 38,970
display(
    retail_df.nlargest(
        20,
        "UnitPrice"
    )[
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "UnitPrice",
            "CustomerID",
            "InvoiceDate"
        ]
    ]
)

# IQR Outlier Detection

for col in ["Quantity", "UnitPrice"]:

    Q1 = retail_df[col].quantile(0.25)
    Q3 = retail_df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = retail_df[
        (retail_df[col] < lower_bound) |
        (retail_df[col] > upper_bound)
    ]

    print(f"\n{col}")
    print("Lower Bound: ", lower_bound)
    print("Upper Bound: ", upper_bound)
    print("Outlier Records:",  len(outliers))
    print(
        "Percentage: ",
        round(len(outliers) / len(retail_df) * 100, 2),
        "%"
    )

    # Post-cleaning validation

print("Final Shape:", retail_df.shape)

print(
    "Missing Values:",
    retail_df.isnull().sum().sum()
)

print(
    "Exact Duplicates:",
    retail_df.duplicated().sum()
)

print(
    "Invalid Dates:",
    retail_df["InvoiceDate"].isnull().sum()
)

print(
    "Missing Customer IDs:",
    retail_df["CustomerID"].isnull().sum()
)

# Post-Cleaning Business Rule Validation

print("=" * 60)
print("POST-CLEANING BUSINESS RULE VALIDATION")
print("=" * 60)


# 1. Missing Value Validation

print("\n1. MISSING VALUES")

print(
    "Total Missing Values:",
    retail_df.isnull().sum().sum()
)

print(
    "Missing CustomerID:",
    retail_df["CustomerID"].isnull().sum()
)

print(
    "Missing StockCode:",
    retail_df["StockCode"].isnull().sum()
)

print(
    "Missing Description:",
    retail_df["Description"].isnull().sum()
)

print(
    "Missing InvoiceDate:",
    retail_df["InvoiceDate"].isnull().sum()
)

print(
    "Missing Country:",
    retail_df["Country"].isnull().sum()
)

# 2. Duplicate Validation

print("\n2. DUPLICATE RECORDS")

exact_duplicates = retail_df.duplicated().sum()

print(
    "Exact Duplicate Rows:",
    exact_duplicates
)

# 3. Quantity Validation

print("\n3. QUANTITY VALIDATION")

negative_quantity = (
    retail_df["Quantity"] < 0
).sum()

zero_quantity = (
    retail_df["Quantity"] == 0
).sum()

positive_quantity = (
    retail_df["Quantity"] > 0
).sum()

print("Negative Quantity:", negative_quantity)
print("Zero Quantity:", zero_quantity)
print("Positive Quantity:", positive_quantity)


# 4. Unit Price Validation

print("\n4. UNIT PRICE VALIDATION")

negative_price = (
    retail_df["UnitPrice"] < 0
).sum()

zero_price = (
    retail_df["UnitPrice"] == 0
).sum()

positive_price = (
    retail_df["UnitPrice"] > 0
).sum()

print("Negative UnitPrice:", negative_price)
print("Zero UnitPrice:", zero_price)
print("Positive UnitPrice:", positive_price)


# 5. Cancellation / Return Validation

print("\n5. CANCELLATION / RETURN VALIDATION")

cancellation_mask = (
    retail_df["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
)

cancelled_count = cancellation_mask.sum()

print(
    "Cancellation Transactions:",
    cancelled_count
)

# Negative quantities belonging to cancellation invoices

negative_with_cancellation = (
    (retail_df["Quantity"] < 0)
    &
    cancellation_mask
).sum()

print(
    "Negative Quantity with Cancellation:",
    negative_with_cancellation
)

# Negative quantities NOT associated with cancellation invoices

negative_without_cancellation = (
    (retail_df["Quantity"] < 0)
    &
    (~cancellation_mask)
).sum()

print(
    "Negative Quantity WITHOUT Cancellation:",
    negative_without_cancellation
)

# 6. Cancellation Consistency

print("\n6. CANCELLATION CONSISTENCY")

cancellation_validation = pd.crosstab(
    cancellation_mask,
    retail_df["Quantity"] < 0,
    rownames=["Cancellation Invoice"],
    colnames=["Negative Quantity"]
)

display(cancellation_validation)


# 7. CustomerID Validation

print("\n7. CUSTOMER ID VALIDATION")

print(
    "Missing CustomerID:",
    retail_df["CustomerID"].isnull().sum()
)

print(
    "Unique Customers:",
    retail_df["CustomerID"].nunique()
)

print(
    "CustomerID Data Type:",
    retail_df["CustomerID"].dtype
)


# 8. Invoice Date Validation

print("\n8. INVOICE DATE VALIDATION")

print(
    "Invalid / Missing InvoiceDate:",
    retail_df["InvoiceDate"].isnull().sum()
)

print(
    "Earliest Transaction:",
    retail_df["InvoiceDate"].min()
)

print(
    "Latest Transaction:",
    retail_df["InvoiceDate"].max()
)


# 9. Text / Categorical Validation

print("\n9. TEXT / CATEGORICAL VALIDATION")

empty_description = (
    retail_df["Description"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

empty_country = (
    retail_df["Country"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print(
    "Empty Descriptions:",
    empty_description
)

print(
    "Empty Country Values:",
    empty_country
)


# 10. Final Dataset Shape

print("\n10. FINAL DATASET")

print(
    "Final Rows:",
    retail_df.shape[0]
)

print(
    "Final Columns:",
    retail_df.shape[1]
)

# ============================================================
# Final Cleaning Validation Summary
# ============================================================

validation_summary = pd.DataFrame({

    "Validation_Check": [
        "Total Missing Values",
        "Missing CustomerID",
        "Missing StockCode",
        "Missing Description",
        "Invalid InvoiceDate",
        "Exact Duplicate Rows",
        "Negative Quantity",
        "Zero Quantity",
        "Negative UnitPrice",
        "Zero UnitPrice",
        "Cancellation Transactions",
        "Negative Quantity With Cancellation",
        "Negative Quantity Without Cancellation",
        "Empty Description",
        "Empty Country"
    ],

    "Record_Count": [
        retail_df.isnull().sum().sum(),
        retail_df["CustomerID"].isnull().sum(),
        retail_df["StockCode"].isnull().sum(),
        retail_df["Description"].isnull().sum(),
        retail_df["InvoiceDate"].isnull().sum(),
        retail_df.duplicated().sum(),
        (retail_df["Quantity"] < 0).sum(),
        (retail_df["Quantity"] == 0).sum(),
        (retail_df["UnitPrice"] < 0).sum(),
        (retail_df["UnitPrice"] == 0).sum(),
        cancellation_mask.sum(),
        (
            (retail_df["Quantity"] < 0)
            & cancellation_mask
        ).sum(),
        (
            (retail_df["Quantity"] < 0)
            & (~cancellation_mask)
        ).sum(),
        empty_description,
        empty_country
    ]
})

display(validation_summary)


#Before-vs-after cleaning summary

cleaning_summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Missing Values",
        "Duplicate Rows"
    ],

    "Before Cleaning": [
        initial_rows,
        initial_missing,
        initial_duplicates
    ],

    "After Cleaning": [
        len(retail_df),
        retail_df.isnull().sum().sum(),
        retail_df.duplicated().sum()
    ]
})

display(cleaning_summary)

#Save the clean dataset
retail_df.to_csv(
    "../Data/online_retail_cleaned.csv",
    index=False
)



Raw Dataset Shape:  (541909, 8)
Initial rows: 541909
Initial columns: 8
Initial missing values: 136534
Initial duplicates: 5268
duplicates removed: 5225
['United Kingdom' 'France' 'Australia' 'Netherlands' 'Germany' 'Norway'
 'EIRE' 'Switzerland' 'Spain' 'Poland' 'Portugal' 'Italy' 'Belgium'
 'Lithuania' 'Japan' 'Iceland' 'Channel Islands' 'Denmark' 'Cyprus'
 'Sweden' 'Austria' 'Israel' 'Finland' 'Greece' 'Singapore' 'Lebanon'
 'United Arab Emirates' 'Saudi Arabia' 'Czech Republic' 'Canada'
 'Unspecified' 'Brazil' 'USA' 'European Community' 'Bahrain' 'Malta' 'RSA']
Missing CustomerID before cleaning: 0
Missing CustomerID after cleaning: 0
Rows removed: 0
Dataset shape after CustomerID cleaning: (401604, 9)
Int64


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TransactionStatus
401131,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446,United Kingdom,Completed
401132,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446,United Kingdom,Cancelled


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,InvoiceDate
401131,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,16446,2011-12-09 09:15:00
37511,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,12346,2011-01-18 10:01:00
374208,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,0.00,13256,2011-11-25 15:57:00
312064,573008,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,4800,0.21,12901,2011-10-27 12:26:00
145108,554868,22197,SMALL POPCORN HOLDER,4300,0.72,13135,2011-05-27 10:52:00
62425,544612,22053,EMPIRE DESIGN ROSETTE,3906,0.82,18087,2011-02-22 10:43:00
191372,560599,18007,ESSENTIAL BALM 3.5g TIN IN ENVELOPE,3186,0.06,14609,2011-07-19 17:04:00
33109,540815,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2.10,15749,2011-01-11 12:55:00
111014,550461,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2.10,15749,2011-04-18 13:20:00
322049,573995,16014,SMALL CHINESE STYLE SCISSOR,3000,0.32,16308,2011-11-02 11:24:00


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,InvoiceDate
401132,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08,16446,2011-12-09 09:27:00
37516,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,12346,2011-01-18 10:17:00
3041,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,0.03,15838,2010-12-02 14:23:00
110949,C550456,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,-3114,2.10,15749,2011-04-18 13:08:00
110948,C550456,21175,GIN + TONIC DIET METAL SIGN,-2000,1.85,15749,2011-04-18 13:08:00
110947,C550456,85123A,WHITE HANGING HEART T-LIGHT HOLDER,-1930,2.55,15749,2011-04-18 13:08:00
158864,C556522,22920,HERB MARKER BASIL,-1515,0.55,16938,2011-06-13 11:21:00
130282,C552995,M,Manual,-1350,0.16,18133,2011-05-12 15:19:00
110946,C550456,47566B,TEA TIME PARTY BUNTING,-1300,2.55,15749,2011-04-18 13:08:00
286720,C570556,20971,PINK BLUE FELT CRAFT TRINKET BOX,-1296,1.06,16029,2011-10-11 11:10:00


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,InvoiceDate
401131,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,16446,2011-12-09 09:15:00
401132,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08,16446,2011-12-09 09:27:00


...


Quantity
Lower Bound:  -13.0
Upper Bound:  27.0
Outlier Records: 26646
Percentage:  6.63 %

UnitPrice
Lower Bound:  -2.5
Upper Bound:  7.5
Outlier Records: 35802
Percentage:  8.91 %
Final Shape: (401604, 9)
Missing Values: 0
Exact Duplicates: 0
Invalid Dates: 0
Missing Customer IDs: 0
POST-CLEANING BUSINESS RULE VALIDATION

1. MISSING VALUES
Total Missing Values: 0
Missing CustomerID: 0
Missing StockCode: 0
Missing Description: 0
Missing InvoiceDate: 0
Missing Country: 0

2. DUPLICATE RECORDS
Exact Duplicate Rows: 0

3. QUANTITY VALIDATION
Negative Quantity: 8872
Zero Quantity: 0
Positive Quantity: 392732

4. UNIT PRICE VALIDATION
Negative UnitPrice: 0
Zero UnitPrice: 40
Positive UnitPrice: 401564

5. CANCELLATION / RETURN VALIDATION
Cancellation Transactions: 8872
Negative Quantity with Cancellation: 8872
Negative Quantity WITHOUT Cancellation: 0

6. CANCELLATION CONSISTENCY


Negative Quantity,False,True
Cancellation Invoice,,
False,392732,0
True,0,8872



7. CUSTOMER ID VALIDATION
Missing CustomerID: 0
Unique Customers: 4372
CustomerID Data Type: Int64

8. INVOICE DATE VALIDATION
Invalid / Missing InvoiceDate: 0
Earliest Transaction: 2010-12-01 08:26:00
Latest Transaction: 2011-12-09 12:50:00

9. TEXT / CATEGORICAL VALIDATION
Empty Descriptions: 0
Empty Country Values: 0

10. FINAL DATASET
Final Rows: 401604
Final Columns: 9


,Validation_Check,Record_Count
0,Total Missing Values,0
1,Missing CustomerID,0
2,Missing StockCode,0
3,Missing Description,0
4,Invalid InvoiceDate,0
5,Exact Duplicate Rows,0
6,Negative Quantity,8872
7,Zero Quantity,0
8,Negative UnitPrice,0
9,Zero UnitPrice,40


,Metric,Before Cleaning,After Cleaning
0,Rows,541909,401604
1,Missing Values,136534,0
2,Duplicate Rows,5268,0
